In [1]:
#!pip install groq --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [2]:
import faiss
import numpy as np
import pandas as pd

from groq import Groq
from sentence_transformers import SentenceTransformer

In [3]:
chunks_df = pd.read_pickle("../data/chunks.pkl")

embeddings = np.load("../data/embeddings.npy")

index = faiss.read_index("../data/faiss_index.bin")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
client = Groq(
    api_key="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"
)

MODEL_NAME = "llama-3.1-8b-instant"

In [5]:
def build_prompt(question, retrieved_chunks):

    context = "\n\n".join(
        retrieved_chunks["chunk_text"].tolist()
    )

    prompt = f"""
You are an AI Teaching Assistant.

Answer ONLY using the provided context.

If the answer is not found in the context,
say exactly:

I couldn't find the answer in the provided course materials.

Context
-------
{context}

Question
--------
{question}

Answer:
"""

    return prompt

In [6]:
def retrieve(query, k=5):

    query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = chunks_df.iloc[indices[0]].copy()

    results["distance"] = distances[0]

    return results

In [7]:
def ask_rag(question, k=5):

    retrieved = retrieve(question, k)

    prompt = build_prompt(question, retrieved)

    response = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2
    )

    answer = response.choices[0].message.content

    return answer, retrieved

In [8]:
from IPython.display import Markdown, display

while True:

    print("=" * 100)

    question = input("🎓 Ask EduRAG: ")

    if question.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye!")
        break

    print("\n🔍 Searching course materials...\n")

    answer, sources = ask_rag(question)

    # Answer
    display(Markdown("## 🤖 Answer"))
    print(answer)

    print("\n" + "=" * 100)

    display(Markdown("## 📚 Sources Used"))

    for i, (_, row) in enumerate(sources.iterrows(), start=1):

        print(f"\n[{i}] {row['course']}")
        print(f"    📄 File : {row['file_name']}")
        print(f"    📖 Page : {row['page']}")
        print(f"    📏 Distance : {row['distance']:.4f}")

        print("-" * 80)

        preview = row["chunk_text"][:250].replace("\n", " ")

        print(preview + "...")

        print("-" * 80)

🎓 Ask EduRAG:  what is NLP?



🔍 Searching course materials...



## 🤖 Answer

NLP (Natural Language Processing) is a branch of AI that gives machines the ability to read, understand, interpret, and generate (automatic processing of) human language. The ultimate goal is to bridge the communication gap between the Computer and Human.



## 📚 Sources Used


[1] AI Tools
    📄 File : Lec5 Naltural Language Processing.pdf
    📖 Page : 3
    📏 Distance : 0.6908
--------------------------------------------------------------------------------
NLP branch of AI that gives machines the ability to read, understand, interpret, and generate (automatic processing of) human language. The ultimate goal is to bridge the communication gap between the Computer and Human. Natural Language Processing ...
--------------------------------------------------------------------------------

[2] AI Tools
    📄 File : Lec5 Naltural Language Processing.pdf
    📖 Page : 13
    📏 Distance : 0.6324
--------------------------------------------------------------------------------
Natural Language Processing NLP Applications...
--------------------------------------------------------------------------------

[3] AI Tools
    📄 File : Lec5 Naltural Language Processing.pdf
    📖 Page : 8
    📏 Distance : 0.6044
-------------------------------------------------------------

🎓 Ask EduRAG:  exit



👋 Goodbye!


In [10]:
import time

start = time.time()

answer, sources = ask_rag(question)

elapsed = time.time() - start

print(f"\n⏱ Response Time: {elapsed:.2f} sec")


⏱ Response Time: 0.15 sec
